# BERT Fine Tuning for Sentiment Analysis (IMDB)

## Objective
The goal of this notebook is to fine-tune a pre-trained BERT model for sentiment classification using the IMDB movie reviews dataset.

## Concepts Covered
- Text preprocessing for NLP
- Tokenization using BERT
- Fine-tuning transformer models
- Model evaluation using classification metrics
- Experimental comparison of different training strategies

## Dataset
The dataset consists of 50,000 movie reviews labeled as positive or negative.

## Workflow
Raw Data > Preprocessing > Tokenization > Dataset > Model Training > Evaluation > Experiments

---

## Import Required Libraries

In [26]:
import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm

---

## Load Dataset

We load the IMDB dataset which contains labeled movie reviews for sentiment analysis.

In [27]:
df = pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.head()
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


---

## Data Exploration

Understanding the structure of the dataset and checking class distribution.

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [29]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

---

## Data Preprocessing

Before training the model, we perform basic preprocessing. Since BERT is a contextual model, only minimal cleaning is applied.

Steps performed:
- Handle missing values
- Convert labels to numeric format
- Clean text by removing unwanted elements

In [30]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [31]:
df.dropna(inplace=True)

Handling missing values ensures that the dataset does not contain null entries, which could cause issues during training.

---

## Convert Labels to Numeric Format

Machine learning models require numeric labels. We convert:
- positive -> 1
- negative -> 0

In [32]:
df['sentiment'].unique()

array(['positive', 'negative'], dtype=object)

In [33]:
df['sentiment'] = df['sentiment'].str.strip().str.lower()

Before label conversion, we normalize the sentiment column by removing extra spaces and converting text to lowercase. This ensures correct mapping to numeric labels.

In [34]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [35]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


This step converts categorical labels into numeric values so that the model can process them during training.

---

## Text Cleaning

We perform minimal preprocessing to remove noise from the text while preserving its meaning.

Steps:
- Convert text to lowercase
- Remove HTML tags
- Remove special characters

In [36]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['review'] = df['review'].apply(clean_text)

In [37]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming tech...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically theres a family where a little boy j...,0
4,petter matteis love in the time of money is a ...,1


Text is cleaned to remove unnecessary elements such as HTML tags and special characters. 
Only basic cleaning is performed because BERT can understand context and language structure effectively.

---

## Data Splitting

The dataset is split into training, validation, and test sets.

- Training set is used to train the model
- Validation set is used to tune the model
- Test set is used for final evaluation

In [38]:
from sklearn.model_selection import train_test_split

# Step 1: Train (70%) and Temp (30%)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.3,
    random_state=42
)

# Step 2: Split Temp into Validation (15%) and Test (15%)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42
)

In [39]:
print(len(train_texts))
print(len(val_texts))
print(len(test_texts))

35000
7500
7500


The dataset is first split into training and temporary sets (70% and 30%).  
The temporary set is further divided equally into validation and test sets.

This ensures proper evaluation and prevents data leakage.

---

## Tokenization

We use the pre-trained BERT tokenizer to convert text into token IDs.  
This step prepares the input data in a format suitable for the BERT model.

In [40]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

To monitor progress and improve usability, we process the data in batches and use a progress bar.

This helps track how much of the data has been processed during tokenization.

In [41]:
def tokenize_in_batches(texts, tokenizer, batch_size=1000):
    encodings = {'input_ids': [], 'attention_mask': []}
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        tokenized = tokenizer(batch, truncation=True, padding=True)
        
        encodings['input_ids'].extend(tokenized['input_ids'])
        encodings['attention_mask'].extend(tokenized['attention_mask'])
    
    return encodings

In [42]:
print("Tokenizing training data...")
train_encodings = tokenize_in_batches(list(train_texts), tokenizer)

print("Tokenizing validation data...")
val_encodings = tokenize_in_batches(list(val_texts), tokenizer)

print("Tokenizing test data...")
test_encodings = tokenize_in_batches(list(test_texts), tokenizer)

Tokenizing training data...


100%|██████████| 35/35 [00:14<00:00,  2.34it/s]


Tokenizing validation data...


100%|██████████| 8/8 [00:02<00:00,  2.69it/s]


Tokenizing test data...


100%|██████████| 8/8 [00:03<00:00,  2.06it/s]


In [43]:
train_encodings.keys()

dict_keys(['input_ids', 'attention_mask'])

The tokenizer converts text into token IDs and generates attention masks.  
These are required inputs for the BERT model.

---

## Creating Dataset

After tokenization, the data is converted into a custom PyTorch dataset.

This step is required because the Trainer expects inputs in a specific format that includes:
- input_ids
- attention_mask
- labels

The dataset class helps organize tokenized data and labels so they can be used during training.

In [44]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [45]:
train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)

---

## Load BERT Model

We load a pre-trained BERT model for sequence classification.

The model already understands language patterns and is fine-tuned on our dataset.

We set `num_labels=2` since this is a binary classification problem (positive and negative).

In [46]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


---

## Training Configuration

We define training parameters such as learning rate, batch size, and number of epochs. These control how the model learns during fine-tuning.

In [47]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2
)

---

## Trainer Setup

The Trainer API simplifies the training process by handling batching, evaluation, and optimization.

In [48]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

---

## Model Training

We start fine-tuning the BERT model on the training dataset.

In [49]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
500,0.619188
1000,0.504376
1500,0.446605
2000,0.434649
2500,0.335226
3000,0.273850
3500,0.270467
4000,0.302952


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4376, training_loss=0.3889424761644883, metrics={'train_runtime': 4105.716, 'train_samples_per_second': 17.049, 'train_steps_per_second': 1.066, 'total_flos': 1.84177738752e+16, 'train_loss': 0.3889424761644883, 'epoch': 2.0})

---

## Evaluation

After training, the model is evaluated on the test dataset.

We use standard classification metrics:
- Accuracy
- Precision
- Recall
- F1 Score

A confusion matrix is also used to visualize prediction performance.

In [50]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(test_labels, preds))
print(confusion_matrix(test_labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

           0       0.93      0.94      0.94      3722
           1       0.94      0.93      0.94      3778

    accuracy                           0.94      7500
   macro avg       0.94      0.94      0.94      7500
weighted avg       0.94      0.94      0.94      7500

[[3514  208]
 [ 264 3514]]


---

## Analysis

The BERT model achieved an accuracy of 94%, indicating strong performance on the IMDB dataset.

The model shows balanced precision and recall across both classes, meaning it does not favor positive or negative predictions.

The confusion matrix shows a low number of misclassifications, suggesting that the model generalizes well.

This performance highlights the effectiveness of transformer-based models like BERT in understanding contextual language patterns compared to traditional NLP methods.

---

## Experiment 1: Freeze BERT Layers

In this experiment, we freeze all BERT layers and train only the classification head.

In [ ]:
from transformers import AutoModelForSequenceClassification

model_frozen = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Freeze all BERT layers
for param in model_frozen.bert.parameters():
    param.requires_grad = False

In [ ]:
# Training Setup
from transformers import TrainingArguments

training_args_exp = TrainingArguments(
    output_dir='./results_exp1',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)

In [ ]:
# Trainer
trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args_exp,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer_frozen.train()

In [ ]:
# Evaluation
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer_frozen.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, preds))
print(confusion_matrix(test_labels, preds))

---

## Experiment 2: Fine-Tuning Last 2 Layers

In this experiment, only the last two layers of BERT are trained while others are frozen.